Imports and Configuration

In [27]:
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Tic-Tac-Toe Environment

In [28]:
class TicTacToe:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = np.zeros(9, dtype=np.int8)
        self.player = 1
        return self

    def copy(self):
        game = TicTacToe()
        game.board = self.board.copy()
        game.player = self.player
        return game

    def legal_moves(self):
        return np.flatnonzero(self.board == 0).tolist()

    def make_move(self, move):
        if self.board[move] != 0:
            raise ValueError("Illegal move")

        self.board[move] = self.player
        self.player *= -1

    def winner(self):
        lines = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),
            (0, 3, 6), (1, 4, 7), (2, 5, 8),
            (0, 4, 8), (2, 4, 6)
        ]

        for a, b, c in lines:
            total = self.board[a] + self.board[b] + self.board[c]

            if total == 3:
                return 1
            if total == -3:
                return -1

        return 0

    def terminal(self):
        return self.winner() != 0 or not np.any(self.board == 0)

    def result(self):
        winner = self.winner()

        if winner != 0:
            return winner

        if not np.any(self.board == 0):
            return 0

        return None

    def render(self):
        symbols = {1: "X", -1: "O", 0: " "}

        for r in range(3):
            print(
                f" {symbols[self.board[3*r]]} |"
                f" {symbols[self.board[3*r+1]]} |"
                f" {symbols[self.board[3*r+2]]}"
            )
            if r < 2:
                print("---+---+---")

    def key(self):
        return tuple(self.board) + (self.player,)

Environment Test

In [29]:
game = TicTacToe()

print("Initial board:")
game.render()

print("\nLegal moves:", game.legal_moves())
print("Current player:", "X" if game.player == 1 else "O")
print("Terminal:", game.terminal())

Initial board:
   |   |  
---+---+---
   |   |  
---+---+---
   |   |  

Legal moves: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Current player: X
Terminal: False


MCTS Node

In [30]:
class Node:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move

        self.children = {}
        self.untried = state.legal_moves()

        self.visits = 0
        self.value = 0.0

    @property
    def q(self):
        return self.value / self.visits if self.visits else 0.0

    @property
    def fully_expanded(self):
        return len(self.untried) == 0

UCT

In [31]:
def uct(child, parent, c=math.sqrt(2)):
    if child.visits == 0:
        return float("inf")

    exploitation = child.q
    exploration = c * math.sqrt(
        math.log(parent.visits) / child.visits
    )

    return exploitation + exploration

Pure MCTS

In [32]:
class MCTS:
    def __init__(self, simulations=1000, c=math.sqrt(2)):
        self.simulations = simulations
        self.c = c

    def select(self, node):
        while node.fully_expanded and not node.state.terminal():
            node = max(
                node.children.values(),
                key=lambda child: uct(child, node, self.c)
            )
        return node

    def expand(self, node):
        move = random.choice(node.untried)
        node.untried.remove(move)

        state = node.state.copy()
        state.make_move(move)

        child = Node(state, node, move)
        node.children[move] = child

        return child

    def rollout(self, state):
        state = state.copy()
        root_player = state.player

        while not state.terminal():
            move = random.choice(state.legal_moves())
            state.make_move(move)

        result = state.result()

        if result == 0:
            return 0

        return 1 if result == root_player else -1

    def backpropagate(self, node, result):
        while node is not None:
            node.visits += 1
            node.value += result

            # Negamax perspective:
            # switch value when moving to opponent's node.
            result = -result
            node = node.parent

    def search(self, state):
        root = Node(state.copy())

        for _ in range(self.simulations):
            node = self.select(root)

            if not node.state.terminal():
                node = self.expand(node)

            result = self.rollout(node.state)
            self.backpropagate(node, result)

        return root

    def choose_move(self, state):
        root = self.search(state)

        return max(
            root.children.values(),
            key=lambda child: child.visits
        ).move

Test Pure MCTS

In [33]:
game = TicTacToe()

mcts = MCTS(simulations=1000)

move = mcts.choose_move(game)

print("Selected move:", move)

Selected move: 7
